# Week 4 — Feature engineering + XGBoost / Random Forest ensemble

Imports (same stack as week 3 plus RF).

In [11]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

Load Adult, clean `?`, binarize target, drop `education` (redundant with `educational-num`).

In [12]:
adult = pd.read_csv("adult.csv").replace("?", np.nan)
adult["income"] = (adult["income"].str.strip() == ">50K").astype(int)
adult = adult.drop(columns=["education"])
adult["gender"] = (adult["gender"] == "Male").astype(int)
adult.head()

,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,0,0,0,30,United-States,0


Stratified 80/20 split (same as week 3).

In [13]:
train_idx, test_idx = train_test_split(
    adult.index, test_size=0.2, stratify=adult["income"], random_state=42
)
train_df, test_df = adult.loc[train_idx].copy(), adult.loc[test_idx].copy()
y_train, y_test = train_df["income"], test_df["income"]
print(train_df.shape, test_df.shape)

(39073, 14) (9769, 14)


Engineered features: log transform, interaction, grouped categories.

In [14]:
def education_tier(n):
    if pd.isna(n): return "unknown"
    n = int(n)
    if n <= 9: return "low"
    if n <= 12: return "mid"
    return "high"

def group_workclass(x):
    if pd.isna(x): return "Unknown"
    if x == "Private": return "Private"
    if x in ("Local-gov", "State-gov", "Federal-gov"): return "Government"
    if "Self-emp" in str(x): return "Self_employed"
    return "Other"

def add_features(df):
    out = df.copy()
    cg = out["capital-gain"].fillna(0)
    cl = out["capital-loss"].fillna(0)
    out["capital_flow_log"] = np.log1p(cg + cl)
    out["education_x_hours"] = out["educational-num"] * out["hours-per-week"]
    out["is_us_native"] = (out["native-country"].fillna("Unknown") == "United-States").astype(int)
    out["education_tier"] = out["educational-num"].map(education_tier)
    out["workclass_group"] = out["workclass"].map(group_workclass)
    return out

train_fe = add_features(train_df)
test_fe = add_features(test_df)
train_fe.head()

,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income,capital_flow_log,education_x_hours,is_us_native,education_tier,workclass_group
34342,71,Private,77253,9,Never-married,Handlers-cleaners,Not-in-family,White,1,0,0,17,United-States,0,0.0,153,1,low,Private
18559,17,Private,329783,6,Never-married,Sales,Other-relative,White,0,0,0,10,United-States,0,0.0,60,1,low,Private
12477,27,Private,91257,9,Married-civ-spouse,Other-service,Husband,White,1,0,0,40,El-Salvador,0,0.0,360,0,low,Private
560,43,Private,125577,9,Separated,Adm-clerical,Unmarried,Black,0,0,0,40,United-States,0,0.0,360,1,low,Private
3427,31,Private,137978,13,Married-civ-spouse,Exec-managerial,Husband,White,1,0,0,40,United-States,0,0.0,520,1,high,Private


Fill remaining categorical NaNs and define the modeling columns.

In [15]:
for c in ["workclass", "marital-status", "occupation", "relationship", "race", "native-country"]:
    train_fe[c] = train_fe[c].fillna("Unknown")
    test_fe[c] = test_fe[c].fillna("Unknown")

num_cols = ["age", "fnlwgt", "educational-num", "capital-gain", "capital-loss",
            "hours-per-week", "gender", "capital_flow_log", "education_x_hours", "is_us_native"]
cat_cols = ["workclass", "marital-status", "occupation", "relationship", "race",
            "native-country", "education_tier", "workclass_group"]

X_train = train_fe[num_cols + cat_cols]
X_test = test_fe[num_cols + cat_cols]
X_train.shape, X_test.shape

((39073, 18), (9769, 18))

XGBoost version of the features — keep categoricals as `category` dtype (native handling).

In [16]:
X_train_xgb = X_train.copy()
X_test_xgb = X_test.copy()
for c in cat_cols:
    X_train_xgb[c] = X_train_xgb[c].astype("category")
    X_test_xgb[c] = X_test_xgb[c].astype("category")

Random Forest version — one-hot encode inside a pipeline (RF needs numeric).

In [17]:
prep_rf = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])
pipe_rf = Pipeline([
    ("prep", prep_rf),
    ("clf", RandomForestClassifier(random_state=42, n_jobs=-1)),
])

Stratified 5-fold CV, scoring on F1 (same choice as week 3).

In [18]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Tune XGBoost — same 5 knobs tuned in week 3.

In [19]:
spw = (y_train == 0).sum() / (y_train == 1).sum()

grid_xgb = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [200, 400],
    "scale_pos_weight": [1.0, spw],
    "subsample": [0.8, 1.0],
}

search_xgb = GridSearchCV(
    XGBClassifier(random_state=42, enable_categorical=True, tree_method="hist"),
    grid_xgb, scoring="f1", cv=cv, n_jobs=-1,
)
search_xgb.fit(X_train_xgb, y_train)
print("XGB best:", search_xgb.best_params_)
print("XGB CV F1:", round(search_xgb.best_score_, 4))

XGB best: {'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 400, 'scale_pos_weight': np.float64(3.1793774735265803), 'subsample': 0.8}
XGB CV F1: 0.7174


Tune Random Forest — trees, depth, leaf size, features.

In [ ]:
grid_rf = {
    "clf__n_estimators": [150, 300],
    "clf__max_depth": [None, 15, 20],
    "clf__min_samples_leaf": [2, 5],
    "clf__max_features": ["sqrt", 0.5],
}

search_rf = GridSearchCV(pipe_rf, grid_rf, scoring="f1", cv=cv, n_jobs=-1)
search_rf.fit(X_train, y_train)
print("RF best:", search_rf.best_params_)
print("RF CV F1:", round(search_rf.best_score_, 4))

RF best: {'clf__max_depth': 20, 'clf__max_features': 0.5, 'clf__min_samples_leaf': 2, 'clf__n_estimators': 300}
RF CV F1: 0.6902


Exception ignored in: <function ResourceTracker.__del__ at 0x106badbc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x10306dbc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1043f9bc0>
Traceback (most recent call last

Default vs tuned — shows tuning moved the needle.

In [22]:
def report(name, model, X_tr, X_te):
    model.fit(X_tr, y_train)
    p = model.predict_proba(X_te)[:, 1]
    y_hat = (p >= 0.5).astype(int)
    print(f"{name}: AUC={roc_auc_score(y_test, p):.4f}  "
          f"acc={accuracy_score(y_test, y_hat):.4f}  F1={f1_score(y_test, y_hat):.4f}")

report("XGB default", XGBClassifier(random_state=42, enable_categorical=True, tree_method="hist"),
       X_train_xgb, X_test_xgb)
report("RF  default", Pipeline([("prep", prep_rf),
                                ("clf", RandomForestClassifier(random_state=42, n_jobs=-1))]),
       X_train, X_test)
report("XGB tuned", search_xgb.best_estimator_, X_train_xgb, X_test_xgb)
report("RF  tuned", search_rf.best_estimator_, X_train, X_test)

XGB default: AUC=0.9276  acc=0.8726  F1=0.7111
RF  default: AUC=0.9071  acc=0.8596  F1=0.6849
XGB tuned: AUC=0.9280  acc=0.8442  F1=0.7201
RF  tuned: AUC=0.9199  acc=0.8707  F1=0.7002


Probability averaging ensemble: equal weights and CV-F1 weighted.

In [23]:
best_xgb = search_xgb.best_estimator_
best_rf = search_rf.best_estimator_

p_xgb = best_xgb.predict_proba(X_test_xgb)[:, 1]
p_rf = best_rf.predict_proba(X_test)[:, 1]

p_eq = 0.5 * p_xgb + 0.5 * p_rf
y_eq = (p_eq >= 0.5).astype(int)
print(f"Ensemble (equal): AUC={roc_auc_score(y_test, p_eq):.4f}  F1={f1_score(y_test, y_eq):.4f}")

wa, wb = search_xgb.best_score_, search_rf.best_score_
w_xgb, w_rf = wa / (wa + wb), wb / (wa + wb)
p_w = w_xgb * p_xgb + w_rf * p_rf
y_w = (p_w >= 0.5).astype(int)
print(f"Ensemble (CV-F1 weighted {w_xgb:.2f}/{w_rf:.2f}): "
      f"AUC={roc_auc_score(y_test, p_w):.4f}  F1={f1_score(y_test, y_w):.4f}")

print("\nEnsemble (equal) report:\n", classification_report(y_test, y_eq, digits=4))

Ensemble (equal): AUC=0.9278  F1=0.7302
Ensemble (CV-F1 weighted 0.51/0.49): AUC=0.9279  F1=0.7314

Ensemble (equal) report:
               precision    recall  f1-score   support

           0     0.9218    0.9004    0.9110      7431
           1     0.7052    0.7571    0.7302      2338

    accuracy                         0.8661      9769
   macro avg     0.8135    0.8287    0.8206      9769
weighted avg     0.8699    0.8661    0.8677      9769



Quick look at feature importances (RF on the OHE space).

In [24]:
rf_model = best_rf.named_steps["clf"]
rf_names = best_rf.named_steps["prep"].get_feature_names_out()
rf_imp = (pd.Series(rf_model.feature_importances_, index=rf_names)
            .sort_values(ascending=False).head(15))
rf_imp

cat__marital-status_Married-civ-spouse    0.177220
num__fnlwgt                               0.094631
num__age                                  0.089336
num__capital_flow_log                     0.089038
num__education_x_hours                    0.084969
num__capital-gain                         0.078151
cat__relationship_Husband                 0.069124
num__educational-num                      0.068465
cat__education_tier_high                  0.038089
num__capital-loss                         0.033383
num__hours-per-week                       0.027670
cat__occupation_Exec-managerial           0.013733
cat__relationship_Wife                    0.011671
cat__education_tier_low                   0.008162
cat__occupation_Prof-specialty            0.006562
dtype: float64

### Evaluation

- **Best single model (test set):** Tuned **XGBoost** had the highest F1 among the two tuned models (**0.7201** vs RF tuned **0.7002**). It also had the best AUC (**0.9280** vs RF **0.9199**). Note: tuning XGB **lowered accuracy** vs the default XGB (**0.8442** vs **0.8726**) while **raising F1**, which is a common trade-off when you tune for the minority class (`scale_pos_weight` ≈ 3.18 in the best grid).

- **Did the ensemble help?** **Yes on F1.** The equal-weight blend reached **F1 = 0.7302** (weighted blend **0.7314**), better than tuned XGB (**0.7201**) and tuned RF (**0.7002**). AUC was essentially flat vs tuned XGB alone (**0.9278–0.9279** vs **0.9280**), so the gain was mainly in the class-1 balance captured by F1.

- **Useful engineered features (from RF importances):** **`capital_flow_log`** and **`education_x_hours`** sit in the top tier with raw **`capital-gain`**, **`age`**, and **`fnlwgt`**, so the log transform and interaction are carrying real signal in the tree model. **`education_tier_high`** also appears in the top 15, which supports bucketing education for interpretability.

- **Different responses:** **RF** ranks **`marital-status_Married-civ-spouse`** and **`relationship_Husband`** very high—sparse OHE levels that interact with how income is labeled in the data. **XGB** uses native categoricals on the same columns without expanding to the full OHE matrix first, so it can learn similar splits with a different internal representation. Both still benefit from **`capital_flow_log`** and **`education_x_hours`** in the shared numeric block.

- **Workflow takeaway:** Keep **row-safe** engineered columns (logs, products, grouped bins) before the split; use **different encodings** per algorithm (category dtype + `enable_categorical` for XGB, OHE in a pipeline for RF). **Average probabilities** when the second model’s errors differ—here it lifted F1 without needing a much better single AUC.